# Kestrel student: QLoRA fine-tune (Kaggle)

Runs `train.py` from the **kestrel-distill-code** dataset on the **kestrel-distill-data** dataset
(`train.jsonl`, `val.jsonl`, `manifest.json` from `kestrel distill build`). All settings are in
`train_config.toml`; nothing is set here. Steps and expected run time: `distill/TRAINING.md`.

Settings needed: **Accelerator: GPU T4** (one is used), **Internet: on** (pip and the model download).
No token is needed (the base model isn't gated). Never paste a token into a cell: use Add-ons > Secrets.

In [ ]:
# Where Kaggle mounts the two input datasets, and where the output goes. Change only if you named them differently.
DATA = "/kaggle/input/kestrel-distill-data"
CODE = "/kaggle/input/kestrel-distill-code"
OUT = "/kaggle/working/run"

In [ ]:
import subprocess
import sys

print(sys.version)
assert sys.version_info >= (3, 11), "trl needs Python 3.11+"
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout or "no GPU: set Accelerator to GPU T4")

In [ ]:
# The versions Kestrel's uv.lock pins (Kaggle's preinstalled CUDA build of torch is kept).
%pip install -q "transformers==5.20.1" "peft==0.21.2" "trl==1.16.0"
%pip install -q "datasets==5.1.0" "accelerate==1.15.0" "bitsandbytes==0.50.2"

In [ ]:
# Optional: a Hugging Face token only raises download rate limits. Add it as a Kaggle secret named HF_TOKEN
# (Add-ons > Secrets, then tick it for this notebook); it is read here and never printed or saved.
import os

try:
    from kaggle_secrets import UserSecretsClient

    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN secret found")
except Exception:
    print("No HF_TOKEN secret (fine: the model isn't gated)")

## GPU smoke test: run this first

The real base model in 4-bit overfits a fixed batch of synthetic examples (`smoke/` in the code dataset):
the loss must fall by half, every example must have trainable tokens, and a greedy generation must make a
tool call that train_format parses. It also prints the library versions next to the pins. A few minutes;
stop here if it says FAILED.

In [ ]:
SMOKE_OUT = "/kaggle/working/gpu-smoke"
!python {CODE}/train.py --gpu-smoke --data {CODE}/smoke --out {SMOKE_OUT} --config {CODE}/train_config.toml

## Training

In [ ]:
!python {CODE}/train.py --data {DATA} --out {OUT} --config {CODE}/train_config.toml

In [ ]:
# Package the adapter and its lineage for download (Output tab > kestrel-adapter.zip). Checkpoints stay behind.
import json
import shutil
from pathlib import Path

lineage = json.loads(Path(OUT, "lineage.json").read_text())
print(json.dumps(lineage["results"] | {"dataset": lineage["dataset"]["manifest_sha256"]}, indent=2, default=str))
package = Path("/kaggle/working/kestrel-adapter")
shutil.rmtree(package, ignore_errors=True)
shutil.copytree(Path(OUT, "adapter"), package / "adapter")
shutil.copy(Path(OUT, "lineage.json"), package / "lineage.json")
print(shutil.make_archive("/kaggle/working/kestrel-adapter", "zip", package))